# ShuffleNet — An Extremely Efficient CNN for Mobile Devices (2017)

_Papers · Computer Vision_

**Make the expensive 1x1 convolution cheap by splitting it into groups, then shuffle channels so the groups can still talk.**

---

This notebook is a practice scaffold for the **ShuffleNet — An Extremely Efficient CNN for Mobile Devices (2017)** lesson. We build the channel-shuffle trick one piece at a time — the FLOP arithmetic, the shuffle op, the ShuffleNet unit, then a toy task that proves why the shuffle matters. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — The shuffle permutation and the FLOP saving, on paper

Before any tensors, we pin down the two ideas with plain arithmetic. **Channel shuffle** is just an index permutation: lay the `g*n` channels into a `(g, n)` grid, transpose, and flatten — so each new group ends up holding one channel from every original group. We also compute the **FLOP cost** of a ResNet bottleneck versus a ShuffleNet one: grouping the 1x1 convolutions divides the `2cm` term by `g`, and the depthwise 3x3 turns `9m^2` into `9m`.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# Channel shuffle as a pure index permutation (no tensors yet): g groups of n channels.
def shuffle_indices(g, n):
    idx = torch.arange(g * n)             # [0, 1, ..., g*n-1]
    idx = idx.view(g, n).t().reshape(-1)  # reshape (g, n) -> transpose -> flatten
    return idx.tolist()

print("shuffle perm g=3,n=4:", shuffle_indices(3, 4))
# shuffle perm g=3,n=4: [0, 4, 8, 1, 5, 9, 2, 6, 10, 3, 7, 11]   <-- matches the worked example

# The worked FLOP block: h=w=28, c=64, m=16, g=4.
def resnet_flops(h, w, c, m):
    return h * w * (2 * c * m + 9 * m * m)

def shufflenet_flops(h, w, c, m, g):
    return h * w * (2 * c * m // g + 9 * m)

h, w, c, m, g = 28, 28, 64, 16, 4
rn = resnet_flops(h, w, c, m)
sn = shufflenet_flops(h, w, c, m, g)
print("ResNet block =", rn, " ShuffleNet block =", sn, " ratio =", round(sn / rn, 4))
# ResNet block = 3411968  ShuffleNet block = 514304  ratio = 0.1507

### Step 2 — The channel-shuffle op on a real tensor

Now the same permutation as a tensor operation over a `(B, C, H, W)` feature map: split the channel axis into `(g, C/g)`, transpose those two axes, and flatten back to `C`. We sanity-check it by running it on channels labelled `0..11` — it must reproduce the index permutation from Step 1, confirming the reshape/transpose really does interleave the groups.

In [ ]:
# The channel-shuffle op on a (B, C, H, W) tensor. Reshape -> transpose -> flatten.
def channel_shuffle(x, groups):
    B, C, H, W = x.shape
    x = x.view(B, groups, C // groups, H, W)   # split channel axis into (g, C/g)
    x = x.transpose(1, 2).contiguous()         # swap the two -> (B, C/g, g, H, W)
    return x.view(B, C, H, W)                  # flatten back to C

# Sanity-check the op reproduces the index permutation on channel ids.
probe = torch.arange(12).float().view(1, 12, 1, 1)   # channels labelled 0..11
shuffled_ids = channel_shuffle(probe, 3).flatten().long().tolist()
print("op on channel ids:", shuffled_ids)
# op on channel ids: [0, 4, 8, 1, 5, 9, 2, 6, 10, 3, 7, 11]   <-- same permutation

### Step 3 — Assemble a ShuffleNet unit

A ShuffleNet unit is a residual block: a grouped 1x1 conv, the channel shuffle, a depthwise 3x3 conv, a second grouped 1x1 conv, then add the input back. The grouped 1x1 convs (`groups=g`) are what make it cheap; the `channel_shuffle` between them is the whole trick — without it the two grouped convs would keep the channel groups permanently isolated. We expose a `shuffle` flag so we can ablate it later.

In [ ]:
# A ShuffleNet unit (residual). groups=g on the 1x1s, depthwise on the 3x3.
class ShuffleUnit(nn.Module):
    def __init__(self, c, m, g, shuffle=True):
        super().__init__()
        self.g = g
        self.shuffle = shuffle
        # pointwise GROUP conv (cheap because of groups=g)
        self.gconv1 = nn.Conv2d(c, m, 1, groups=g, bias=False)
        self.bn1 = nn.BatchNorm2d(m)
        # 3x3 depthwise conv (groups=m -> one filter per channel)
        self.dwconv = nn.Conv2d(m, m, 3, padding=1, groups=m, bias=False)
        self.bn2 = nn.BatchNorm2d(m)
        # second pointwise GROUP conv back up to c channels
        self.gconv2 = nn.Conv2d(m, c, 1, groups=g, bias=False)
        self.bn3 = nn.BatchNorm2d(c)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        out = self.relu(self.bn1(self.gconv1(x)))   # pointwise GROUP conv
        if self.shuffle:
            out = channel_shuffle(out, self.g)      # <-- the whole trick (ablate by skipping)
        out = self.bn2(self.dwconv(out))            # 3x3 depthwise
        out = self.bn3(self.gconv2(out))            # pointwise GROUP conv
        return self.relu(out + x)                   # residual + ReLU

### Step 4 — Prove the shuffle matters on a cross-group task

We build a toy task whose label *requires* combining information from two different channel groups: it compares the mean of channel 0 (group 0) against channel 4 (group 2). A net whose groups stay isolated literally cannot see both at once. We train two identical nets — one with the shuffle on, one off — and compare test accuracy: the shuffled net should learn the task while the un-shuffled one lags.

In [ ]:
# Toy task that REQUIRES cross-group info: the label depends on a comparison between
# channels that the grouping puts in different groups (so within-group-only nets are blocked).
g = 4
torch.manual_seed(1)
Bn, c, m, H, W, K = 600, 8, 16, 8, 8, 2   # c=8 channels -> g=4 groups of 2
gen = torch.Generator().manual_seed(2)
X = torch.randn(Bn, c, H, W, generator=gen)

# signal: mean of channel 0 (group 0) vs channel 4 (group 2) -- different groups must be compared
sig = X[:, 0].mean(dim=(1, 2)) - X[:, 4].mean(dim=(1, 2))
y = (sig > 0).long()
Xtr, ytr = X[:480], y[:480]
Xte, yte = X[480:], y[480:]

class Net(nn.Module):
    def __init__(self, shuffle):
        super().__init__()
        self.u1 = ShuffleUnit(c, m, g, shuffle=shuffle)
        self.u2 = ShuffleUnit(c, m, g, shuffle=shuffle)
        self.head = nn.Linear(c, K)
    def forward(self, x):
        pooled = self.u2(self.u1(x)).mean(dim=(2, 3))   # global average pool over H, W
        return self.head(pooled)

def train(net, epochs=150, lr=0.05):
    torch.manual_seed(0)
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    lf = nn.CrossEntropyLoss()
    for _ in range(epochs):
        net.train()
        opt.zero_grad()
        loss = lf(net(Xtr), ytr)
        loss.backward()
        opt.step()
    net.eval()
    with torch.no_grad():
        return (net(Xte).argmax(1) == yte).float().mean().item()

acc_on = train(Net(shuffle=True))
acc_off = train(Net(shuffle=False))
print("\ntest accuracy   with shuffle = %.3f   without shuffle = %.3f" % (acc_on, acc_off))
# With shuffle the net can combine channels across groups and learns the task; without shuffle
# the groups stay isolated and it lags. (Our small run, not the paper's reported number.)

## Visualize the data & results

_Does channel shuffle let a grouped-conv unit combine information across groups — and does removing it block that?_

### Step 1 — Rebuild the unit and the cross-group task

This panel re-runs the experiment self-contained (so it works even if you jump straight here). We redefine the channel-shuffle op, the `ShuffleUnit`, and the same toy task whose label compares channel 0 against channel 4 — a signal split across two groups, so only a net that can mix groups can read it.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# Channel shuffle: reshape (g, C/g) -> transpose -> flatten.
def channel_shuffle(x, groups):
    B, C, H, W = x.shape
    x = x.view(B, groups, C // groups, H, W)
    x = x.transpose(1, 2).contiguous()
    return x.view(B, C, H, W)

class ShuffleUnit(nn.Module):
    def __init__(s, c, m, g, shuffle=True):
        super().__init__()
        s.g = g
        s.shuffle = shuffle
        s.g1 = nn.Conv2d(c, m, 1, groups=g, bias=False)
        s.b1 = nn.BatchNorm2d(m)
        s.dw = nn.Conv2d(m, m, 3, padding=1, groups=m, bias=False)
        s.b2 = nn.BatchNorm2d(m)
        s.g2 = nn.Conv2d(m, c, 1, groups=g, bias=False)
        s.b3 = nn.BatchNorm2d(c)
    def forward(s, x):
        o = torch.relu(s.b1(s.g1(x)))
        if s.shuffle:
            o = channel_shuffle(o, s.g)
        o = s.b2(s.dw(o))
        o = s.b3(s.g2(o))
        return torch.relu(o + x)

# Toy task: label compares channel 0 (group 0) vs channel 4 (group 2) -> needs cross-group info.
g, c, m, H, W, K = 4, 8, 16, 8, 8, 2
gen = torch.Generator().manual_seed(2)
X = torch.randn(600, c, H, W, generator=gen)
y = ((X[:, 0].mean(dim=(1, 2)) - X[:, 4].mean(dim=(1, 2))) > 0).long()
Xtr, ytr = X[:480], y[:480]
Xte, yte = X[480:], y[480:]

class Net(nn.Module):
    def __init__(s, shuffle):
        super().__init__()
        s.u1 = ShuffleUnit(c, m, g, shuffle)
        s.u2 = ShuffleUnit(c, m, g, shuffle)
        s.head = nn.Linear(c, K)
    def forward(s, x):
        return s.head(s.u2(s.u1(x)).mean(dim=(2, 3)))

### Step 2 — Track accuracy over training and the FLOP bars

We train both nets and record test accuracy every 15 epochs, so you can watch the shuffled net pull ahead. Then we print the FLOP cost of a ResNet, a ResNeXt, and a ShuffleNet block at the same shape — showing how grouping and the depthwise conv compound into a large saving.

In [ ]:
def train(net, epochs=150, lr=0.05):
    torch.manual_seed(0)
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    lf = nn.CrossEntropyLoss()
    curve = []
    for e in range(epochs):
        net.train()
        opt.zero_grad()
        lf(net(Xtr), ytr).backward()
        opt.step()
        if e % 15 == 0 or e == epochs - 1:
            net.eval()
            with torch.no_grad():
                acc = round((net(Xte).argmax(1) == yte).float().mean().item(), 3)
            curve.append((e, acc))
    return curve

print("with shuffle:   ", train(Net(True)))
print("without shuffle:", train(Net(False)))

# FLOP bars: resnet=hw(2cm+9m^2), resnext=hw(2cm+9m^2/g), shufflenet=hw(2cm/g+9m)
#            at h=w=28, c=64, m=16, g=4.
flop_table = [
    ("ResNet", 28 * 28 * (2 * 64 * 16 + 9 * 16 * 16)),
    ("ResNeXt", 28 * 28 * (2 * 64 * 16 + 9 * 16 * 16 // 4)),
    ("ShuffleNet", 28 * 28 * (2 * 64 * 16 // 4 + 9 * 16)),
]
for name, f in flop_table:
    print("%-10s %d" % (name, f))
# with shuffle reaches ~0.98, without ~0.65 -- our small run, not the paper's number.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The shuffle ablation. You have a working ShuffleNet unit. Build a second copy with the
            channel_shuffle call removed (two grouped $1\times1$ convs now stack with the depthwise
            in between but no cross-group mixing), and train both on a task that needs information from different
            channel groups to combine. What happens to accuracy, and what does it demonstrate?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Remove only the channel_shuffle(x, g) line; keep the group count, widths, depthwise conv, optimizer, data, and seed identical. — _An honest ablation changes exactly one thing &mdash; shuffle present vs absent &mdash; so any gap is attributable to it._
- Use a task whose label depends on combining features that the grouping splits across groups (e.g. a relation between channels that fall in different groups). — _If the task only needs within-group information, the shuffle wouldn't matter; the ablation must stress cross-group flow._
- Train both and compare test accuracy (and, optionally, watch the loss curves). — _The shuffled unit can route information across groups; the un-shuffled one is structurally blocked from it._

**Answer:** The shuffled unit reaches clearly higher accuracy; the un-shuffled one plateaus lower because
                 its grouped $1\times1$ convs keep the channel groups isolated &mdash; no path lets a feature in one
                 group influence an output in another. Since the two units are identical except for the shuffle,
                 this isolates the channel-shuffle permutation as the cause, the same qualitative result as
                 Table 3 (37.6% &rarr; 32.4% error with shuffle, $g{=}8$). The CODEVIZ panel shows this
                 contrast on our toy task.

</details>

**Problem 2.** A bottleneck has $h=w=14$, $c=128$, $m=32$. Compute the ResNet cost $hw(2cm+9m^2)$ and the ShuffleNet
            cost $hw(\tfrac{2cm}{g}+9m)$ with $g=8$, and the ratio. Which term dominates the saving?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- $hw=196$, $2cm=2\cdot128\cdot32=8192$, $9m^2=9\cdot1024=9216$. ResNet $=196(8192+9216)=196\cdot17408$. — _Plug into $hw(2cm+9m^2)$; note the two terms are comparable here._
- ShuffleNet: $\tfrac{2cm}{g}=8192/8=1024$, $9m=288$. $=196(1024+288)=196\cdot1312$. — _Grouping divides the $8192$ by $g=8$; depthwise turns $9216$ into $288$._
- Ratio $=1312/17408\approx0.0754$. — _Both heavy terms shrank, so the block is about 13x cheaper._

**Answer:** ResNet $=196\cdot17408=3{,}411{,}968$; ShuffleNet $=196\cdot1312=257{,}152$; ratio
                 $\approx0.075$, i.e. about 13x cheaper. Both savings matter, but the depthwise step is the
                 bigger lever here: it collapsed $9m^2=9216$ to $9m=288$ (a 32x drop on that term), while grouping
                 cut the $1\times1$ term by $g=8$. With wider bottlenecks ($m$ large relative to $c$) the depthwise
                 term saves even more; with $m$ small the grouped $1\times1$ dominates.

</details>

**Problem 3.** For channel shuffle with $g=2$ groups of $n=3$ channels (6 channels, $[0,1,2]$ and $[3,4,5]$), work
            out the output permutation by hand, then say which original groups each new group draws from.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Reshape $[0,1,2,3,4,5]$ to the $(g,n)=(2,3)$ grid $\begin{smallmatrix}0&1&2\\3&4&5\end{smallmatrix}$. — _Row $i$ is original group $i$._
- Transpose to $(n,g)=(3,2)$: $\begin{smallmatrix}0&3\\1&4\\2&5\end{smallmatrix}$. — _Row $j$ now holds the $j$-th channel of every group._
- Flatten: $[0,3,1,4,2,5]$; slice into new groups of 3: $[0,3,1]$ and $[4,2,5]$. — _The next group conv reads these as its two groups._

**Answer:** The permutation is $[0,3,1,4,2,5]$. New group 0 is $[0,3,1]$ and new group 1 is
                 $[4,2,5]$ &mdash; each new group contains channels from both original groups (e.g. group 0
                 has $0,1$ from original group 0 and $3$ from original group 1). That is exactly why the
                 next grouped convolution can now mix information that was previously trapped within a single group.

</details>